# 🦠 Nipah Virus — Named Entity Recognition (NER)

---

| | |
|---|---|
| **Mata Kuliah** | COMP6885001 — Natural Language Processing |
| **Topik** | Information Extraction menggunakan spaCy Rule-Based NER |
| **Entitas** | `DISEASE`, `SYMPTOM`, `LOCATION` |
| **Dataset** | Web scraping berita + dataset manual (300 kalimat) |
| **Model** | spaCy EntityRuler (rule-based, CPU-friendly) |

---

## 📌 Latar Belakang

Virus Nipah (NiV) adalah virus zoonosis berbahaya yang pertama kali ditemukan di Malaysia tahun 1998. Penyakit ini dapat menyebabkan ensefalitis (radang otak) hingga kematian dengan tingkat fatalitas mencapai 40–75%.

Proyek ini membangun sistem **Named Entity Recognition (NER)** untuk mengekstrak informasi penting dari teks berita dan artikel tentang virus Nipah secara otomatis:
- 🔴 **DISEASE** — nama penyakit/virus (contoh: *virus Nipah, ensefalitis, NiV*)
- 🟡 **SYMPTOM** — gejala klinis (contoh: *demam, kejang, sesak napas*)
- 🔵 **LOCATION** — lokasi kejadian (contoh: *Malaysia, Kerala, Bangladesh*)

---

## 🗂️ Alur Pipeline

```
1. Install & Import
2. Web Scraping Berita  ──┐
3. Load Dataset Lama   ──┴──▶  4. Gabung & Deduplikasi
                                        │
                                        ▼
                               5. Preprocessing
                                        │
                                        ▼
                           6. NER spaCy Rule-Based
                                        │
                                        ▼
                           7. Evaluasi (P / R / F1)
                                        │
                                        ▼
                          8. Simpan Model & Dataset
```

---
## 1. Install & Import

Library yang digunakan:

| Library | Fungsi |
|---|---|
| `spacy` | NLP pipeline & EntityRuler untuk rule-based NER |
| `en_core_web_sm` | Model bahasa Inggris kecil dari spaCy |
| `requests` + `beautifulsoup4` | Web scraping artikel berita |
| `pandas` | Manipulasi dan penyimpanan dataset |
| `re` | Preprocessing teks dengan regex |

In [1]:
!pip install -q spacy requests beautifulsoup4 lxml
!python -m spacy download en_core_web_sm -q
!pip install -q datasets
!pip install -q wikipedia-api
print('Semua library berhasil diinstall!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 65.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 2.8 MB/s eta 0:00:00
Semua library berhasil diinstall!


In [2]:
import os
import pandas as pd
import requests
import re
import time
import spacy
import wikipediaapi
import unicodedata

from collections import Counter
from datasets import load_dataset
from spacy.pipeline import EntityRuler
from bs4 import BeautifulSoup
from collections import defaultdict
from spacy import displacy


print('Semua library berhasil diimport!')

Semua library berhasil diimport!


---
## 2. Web Scraping Berita

Untuk memperbanyak data, dilakukan **web scraping** otomatis dari Google News RSS Feed.

### Mengapa Web Scraping?
Dataset yang dikumpulkan secara manual memiliki keterbatasan jumlah dan variasi kalimat. Web scraping memungkinkan kita mendapatkan data yang lebih banyak, lebih beragam, dan mencerminkan penggunaan bahasa yang nyata (*real-world text*).

### Sumber Data
- **Google News RSS** — snippet judul dan deskripsi berita dari berbagai media
- Query menggunakan kata kunci **Bahasa Indonesia dan Inggris** untuk cakupan yang lebih luas

> ⚠️ **Catatan etika:** Data hanya mengambil snippet publik dari RSS feed (bukan full article) sehingga tidak melanggar *terms of service*.

In [3]:
def scrape_google_news(query, max_articles=20):
    """
    Scraping snippet berita dari Google News RSS Feed.
    - Tidak memerlukan API key
    - Gratis dan legal (mengakses feed publik)
    """
    query_encoded = query.replace(' ', '+')
    url = f'https://news.google.com/rss/search?q={query_encoded}&hl=id&gl=ID&ceid=ID:id'
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'xml')
        items = soup.find_all('item')
    except Exception as e:
        print(f'  ⚠️  Gagal scraping "{query}": {e}')
        return []

    results = []
    for item in items[:max_articles]:
        title = item.find('title').text if item.find('title') else ''
        desc  = item.find('description').text if item.find('description') else ''
        raw   = title + '. ' + desc
        clean = BeautifulSoup(raw, 'html.parser').get_text()
        clean = re.sub(r'\s+', ' ', clean).strip()
        if len(clean) > 30:
            results.append(clean)
    return results

In [4]:
# Daftar query — kombinasi Bahasa Indonesia & Inggris
queries = [
    'virus nipah',
    'nipah virus outbreak',
    'virus nipah gejala',
    'nipah virus symptoms',
    'wabah nipah malaysia india',
    'nipah encephalitis',
    'virus nipah penularan kelelawar',
]

scraped_texts = []
for q in queries:
    texts = scrape_google_news(q, max_articles=20)
    scraped_texts.extend(texts)
    print(f'  ✔ Query "{q}" → {len(texts)} artikel')
    time.sleep(1)  # jeda agar tidak diblokir server

print(f'\n📰 Total teks terkumpul dari scraping: {len(scraped_texts)}')

  ✔ Query "virus nipah" → 20 artikel
  ✔ Query "nipah virus outbreak" → 20 artikel
  ✔ Query "virus nipah gejala" → 20 artikel
  ✔ Query "nipah virus symptoms" → 20 artikel
  ✔ Query "wabah nipah malaysia india" → 20 artikel
  ✔ Query "nipah encephalitis" → 9 artikel
  ✔ Query "virus nipah penularan kelelawar" → 20 artikel

📰 Total teks terkumpul dari scraping: 129


In [5]:
def split_sentences(texts):
    """
    Memecah teks panjang menjadi kalimat-kalimat individual.
    Kalimat kurang dari 20 karakter dibuang (biasanya noise).
    """
    sentences = []
    for text in texts:
        parts = re.split(r'(?<=[.!?])\s+', text)
        for part in parts:
            part = part.strip()
            if len(part) > 20:
                sentences.append(part)
    return sentences

scraped_sentences = split_sentences(scraped_texts)
print(f'🔢 Total kalimat dari scraping: {len(scraped_sentences)}')
print('\nContoh kalimat hasil scraping:')
for s in scraped_sentences[:3]:
    print(f'  - {s}')

🔢 Total kalimat dari scraping: 278

Contoh kalimat hasil scraping:
  - Apa Itu Virus Nipah yang Berpotensi Jadi Ancaman Kesehatan Global?
  - Ini Gejala dan Bahayanya - Mitra Keluarga.
  - Apa Itu Virus Nipah yang Berpotensi Jadi Ancaman Kesehatan Global?


---
## 3. Load Dataset yang Lama (Manual)

Dataset ini berisi **300 kalimat** yang dikumpulkan secara manual dari berbagai artikel dan berita tentang virus Nipah. Dataset ini akan digabungkan dengan hasil scraping untuk membentuk dataset yang lebih lengkap.

> 📁 `Dataset Nipah.csv`

In [6]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive berhasil terhubung!')

Mounted at /content/drive
Google Drive berhasil terhubung!


In [7]:
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if 'nipah' in file.lower() or 'Nipah' in file:
            print(os.path.join(root, file))

/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/nipah_NER.ipynb
/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset Nipah.gsheet
/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/Dataset Nipah.csv
/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/Dataset_Nipah_Final.csv


In [8]:
PATH_DATASET= '/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/Dataset Nipah.csv'

In [9]:
# Sesuaikan path ini dengan lokasi file di Google Drive kamu
data = []
with open(PATH_DATASET, 'r', encoding='latin1') as f:
    f.readline()  # skip baris header
    for line in f:
        line = line.strip()
        if line:
            parts = line.split(',', 1)
            if len(parts) == 2:
                data.append(parts[1])

df_lama = pd.DataFrame({'text': data})
print(f'Dataset sebelumnya berhasil diload: {len(df_lama)} kalimat')
df_lama.head(5)

Dataset sebelumnya berhasil diload: 300 kalimat


,text
0,Virus nipah (NiV) adalah virus zoonosis yang b...
1,Pertama kali diidentifikasi pada tahun 1998 di...
2,Menurut data dari World Health Organization (W...
3,"People with infection can develop a fever, and..."
4,Fruit bats of the Pteropodidae family are the ...


---
## Load Dataset Kaggle — Disease Symptom Description

Dataset ini didownload dari **Kaggle** (`itachi9604/disease-symptom-description-dataset`) dan disimpan di Google Drive.

File yang digunakan: `symptom_Description.csv` — berisi nama penyakit dan deskripsi gejala dalam bentuk kalimat.

> 📁 Path: `.../Dataset/Kaggle/symptom_Description.csv`

In [10]:
PATH_KAGGLE = '/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/Kaggle/symptom_Description.csv'

df_kaggle = pd.read_csv(PATH_KAGGLE)
print(f'Kolom tersedia: {df_kaggle.columns.tolist()}')
print(f'Total baris   : {len(df_kaggle)}')
df_kaggle.head(5)

Kolom tersedia: ['Disease', 'Description']
Total baris   : 41


,Disease,Description
0,Drug Reaction,An adverse drug reaction (ADR) is an injury ca...
1,Malaria,An infectious disease caused by protozoan para...
2,Allergy,An allergy is an immune system response to a f...
3,Hypothyroidism,"Hypothyroidism, also called underactive thyroi..."
4,Psoriasis,Psoriasis is a common skin disorder that forms...


In [11]:
# Ambil kolom deskripsi
desc_col = [c for c in df_kaggle.columns if 'escri' in c or 'desc' in c.lower()][0]
print(f'Kolom deskripsi: {desc_col}')

kaggle_sentences = split_sentences(df_kaggle[desc_col].dropna().astype(str).tolist())
df_kaggle_final = pd.DataFrame({'text': kaggle_sentences})

print(f'Total kalimat dari Kaggle: {len(df_kaggle_final)}')
df_kaggle_final.head(5)

Kolom deskripsi: Description
Total kalimat dari Kaggle: 105


,text
0,An adverse drug reaction (ADR) is an injury ca...
1,ADRs may occur following a single dose or prol...
2,An infectious disease caused by protozoan para...
3,Falciparum malaria is the most deadly type.
4,An allergy is an immune system response to a f...


---
## Web Scraping Wikipedia

Wikipedia menyediakan artikel lengkap tentang Virus Nipah dalam **Bahasa Indonesia dan Inggris**.

Kelebihan sumber ini:
- Konten terstruktur dan terverifikasi secara ensiklopedi
- Mencakup informasi medis, gejala, lokasi wabah, dan sejarah penyebaran
- Kredibel untuk disebutkan sebagai sumber di laporan akademis

> Scraping dilakukan menggunakan library `wikipedia-api`.

In [12]:
def scrape_wikipedia(judul, bahasa='id'):
    """
    Scraping artikel Wikipedia dan memecahnya menjadi kalimat.
    bahasa: 'id' = Indonesia, 'en' = Inggris
    """
    wiki = wikipediaapi.Wikipedia(language=bahasa, user_agent='NipahNER/1.0')
    page = wiki.page(judul)
    if not page.exists():
        print(f'  Halaman "{judul}" tidak ditemukan')
        return []
    sentences = split_sentences([page.text])
    print(f'  ✔ Wikipedia [{bahasa}] "{judul}" → {len(sentences)} kalimat')
    return sentences

wiki_sentences = []
wiki_sentences += scrape_wikipedia('Virus Nipah', 'id')
wiki_sentences += scrape_wikipedia('Nipah virus infection', 'en')
wiki_sentences += scrape_wikipedia('Nipah virus', 'en')

df_wiki = pd.DataFrame({'text': wiki_sentences})
print(f'\nTotal kalimat dari Wikipedia: {len(df_wiki)}')
df_wiki.head(5)

  ✔ Wikipedia [id] "Virus Nipah" → 12 kalimat
  ✔ Wikipedia [en] "Nipah virus infection" → 119 kalimat
  ✔ Wikipedia [en] "Nipah virus" → 121 kalimat

Total kalimat dari Wikipedia: 252


,text
0,Virus Nipah yang teridentifikasi pada tahun 19...
1,"Di Singapura, 11 kasus termasuk satu kematian ..."
2,Virus Nipah diklasifikasikan oleh CDC sebagai ...
3,Kemunculan\nVirus Nipah telah diisolasi dari k...
4,lylei dan kelelawar daun bulat Horsfield (Hipp...


---
##  Web Scraping WHO & CDC

Halaman resmi **WHO** dan **CDC** menyediakan informasi medis yang akurat dan terverifikasi tentang Virus Nipah.

| Sumber | URL |
|---|---|
| WHO | https://www.who.int/news-room/fact-sheets/detail/nipah-virus |
| CDC | https://www.cdc.gov/nipah/about/index.html |

> Konten dari situs resmi organisasi kesehatan dunia memperkuat kualitas dan kredibilitas dataset.

In [13]:
def scrape_website(url, tag='p'):
    """
    Scraping paragraf teks dari website.
    tag: HTML tag yang berisi teks utama (biasanya 'p')
    """
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(resp.content, 'html.parser')
        paragraphs = [p.get_text() for p in soup.find_all(tag) if len(p.get_text()) > 30]
        sentences = split_sentences(paragraphs)
        print(f'  ✔ {url[:55]}... → {len(sentences)} kalimat')
        return sentences
    except Exception as e:
        print(f'  ⚠️  Gagal scraping {url}: {e}')
        return []

who_cdc_sentences = []
who_cdc_sentences += scrape_website('https://www.who.int/news-room/fact-sheets/detail/nipah-virus')
who_cdc_sentences += scrape_website('https://www.cdc.gov/nipah/about/index.html')

df_who_cdc = pd.DataFrame({'text': who_cdc_sentences})
print(f'\n✅ Total kalimat dari WHO + CDC: {len(df_who_cdc)}')
df_who_cdc.head(5)

  ✔ https://www.who.int/news-room/fact-sheets/detail/nipah-... → 57 kalimat
  ✔ https://www.cdc.gov/nipah/about/index.html... → 9 kalimat

✅ Total kalimat dari WHO + CDC: 66


,text
0,"Nipah virus is a zoonotic virus, usually trans..."
1,Nipah virus was first identified in 1998 durin...
2,"In 1999, an outbreak was reported in Singapore..."
3,No new outbreaks have been reported from Malay...
4,"In 2001, Nipah virus infection outbreaks were ..."


---
## 4. Gabungkan & Deduplikasi Dataset

Seluruh sumber data digabungkan menjadi satu dataset komprehensif:

| No | Sumber | Keterangan |
|---|---|---|
| 1 | Dataset manual | 300 kalimat dari berita |
| 2 | Google News RSS | Scraping berita terkini |
| 3 | Kaggle | Deskripsi gejala penyakit |
| 4 | Wikipedia | Artikel Virus Nipah ID + EN |
| 5 | WHO + CDC | Halaman resmi organisasi kesehatan dunia |

Setelah digabung, dilakukan **deduplikasi** untuk menghapus kalimat yang sama persis.

In [14]:
df_scraping = pd.DataFrame({'text': scraped_sentences})

# Gabungkan semua 5 sumber
df_gabung = pd.concat([
    df_lama,          # 1. Dataset manual
    df_scraping,      # 2. Google News scraping
    df_kaggle_final,  # 3. Kaggle disease symptom
    df_wiki,          # 4. Wikipedia
    df_who_cdc,       # 5. WHO + CDC
], ignore_index=True)

print(f'\U0001f4ca Jumlah sebelum deduplikasi : {len(df_gabung)}')
print(f'   1. Dataset manual           : {len(df_lama)}')
print(f'   2. Google News scraping      : {len(df_scraping)}')
print(f'   3. Kaggle disease symptom    : {len(df_kaggle_final)}')
print(f'   4. Wikipedia                 : {len(df_wiki)}')
print(f'   5. WHO + CDC                 : {len(df_who_cdc)}')

# Deduplikasi & bersihkan
df_gabung['text'] = df_gabung['text'].str.strip()
df_gabung = df_gabung.drop_duplicates(subset='text')
df_gabung = df_gabung[df_gabung['text'].str.len() > 20]
df_gabung = df_gabung.reset_index(drop=True)
df_gabung['id'] = df_gabung.index + 1
df_gabung = df_gabung[['id', 'text']]

print(f'\nJumlah setelah deduplikasi  : {len(df_gabung)} kalimat')
df_gabung.head()

📊 Jumlah sebelum deduplikasi : 1001
   1. Dataset manual           : 300
   2. Google News scraping      : 278
   3. Kaggle disease symptom    : 105
   4. Wikipedia                 : 252
   5. WHO + CDC                 : 66

Jumlah setelah deduplikasi  : 871 kalimat


,id,text
0,1,Virus nipah (NiV) adalah virus zoonosis yang b...
1,2,Pertama kali diidentifikasi pada tahun 1998 di...
2,3,Menurut data dari World Health Organization (W...
3,4,"People with infection can develop a fever, and..."
4,5,Fruit bats of the Pteropodidae family are the ...


---
## 5. Preprocessing

Preprocessing membersihkan teks sebelum diproses model NER.

Tahapan preprocessing:
1. **Hapus karakter khusus** — simbol aneh yang tidak relevan
2. **Normalisasi whitespace** — hapus spasi berlebih, tab, dan newline
3. **Strip** — hapus spasi di awal dan akhir kalimat

> Khusus NER rule-based, kita **tidak melakukan lowercase** karena pola seperti `virus Nipah` bergantung pada kapitalisasi.

In [15]:
def preprocess(text):
    # Normalisasi karakter unicode yang rusak
    text = unicodedata.normalize('NFKD', text)
    text = text.encode('ascii', 'ignore').decode('ascii')

    # Hapus karakter noise
    text = re.sub(r'[^\w\s.,!?;:\-\'"()/]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_gabung['text_clean'] = df_gabung['text'].apply(preprocess)

print('Preprocessing selesai!')
# Cari kalimat yang benar-benar berubah setelah preprocessing
for i in range(len(df_gabung)):
    before = df_gabung["text"].iloc[i]
    after  = df_gabung["text_clean"].iloc[i]
    if before != after:
        print(f'Index  : {i}')
        print(f'Sebelum: {before}')
        print(f'Sesudah: {after}')
        break

Preprocessing selesai!
Index  : 9
Sebelum: The incubation period â that is the time from infection to the onset of symptoms â ranges from 3 to 14 days.
Sesudah: The incubation period a that is the time from infection to the onset of symptoms a ranges from 3 to 14 days.


---
## 6. NER — spaCy Rule-Based

### Apa itu Rule-Based NER?
Rule-based NER bekerja dengan **mendefinisikan pola secara manual**. Ketika model menemukan teks yang cocok dengan pola, teks tersebut langsung diberi label entitas.

**Kelebihan untuk kasus Nipah:**
- ✅ Tidak butuh ribuan data latih berlabel
- ✅ Mudah diinterpretasi dan dijelaskan
- ✅ Ringan — jalan di CPU, tidak perlu GPU
- ✅ Cocok untuk domain spesifik dengan kosakata terbatas

### Kategori Entitas

| Label | Deskripsi | Contoh |
|---|---|---|
| `DISEASE` | Nama penyakit atau virus | virus Nipah, NiV, ensefalitis |
| `SYMPTOM` | Gejala klinis | demam, kejang, sesak napas |
| `LOCATION` | Lokasi kejadian wabah | Malaysia, Kerala, Bangladesh |

In [29]:
nlp = spacy.load('en_core_web_sm')
nlp.remove_pipe('ner')  # hapus NER default, ganti dengan rule-based

ruler = nlp.add_pipe('entity_ruler', config={"overwrite_ents": True, "phrase_matcher_attr": "LOWER"})



patterns = [
    # ── DISEASE ──────────────────────────────────────────
    {"label": "DISEASE", "pattern": "Nipah"},
    {"label": "DISEASE", "pattern": "nipah"},
    {"label": "DISEASE", "pattern": "virus Nipah"},
    {"label": "DISEASE", "pattern": "virus nipah"},
    {"label": "DISEASE", "pattern": "NiV"},

    {"label": "DISEASE", "pattern": "Nipah virus"},
    {"label": "DISEASE", "pattern": "ensefalitis"},
    {"label": "DISEASE", "pattern": "encephalitis"},
    {"label": "DISEASE", "pattern": [{"LOWER": "nipah"}, {"LOWER": "virus"}]},
    {"label": "DISEASE", "pattern": [{"LOWER": "encephalitis"}]},
    {"label": "DISEASE", "pattern": [{"LOWER": "ensefalitis"}]},
    {"label": "DISEASE", "pattern": "NiV"},
    {"label": "DISEASE", "pattern": [{"LOWER": "niv"}]},



    # ── SYMPTOM — Bahasa Indonesia ────────────────────────
    {"label": "SYMPTOM", "pattern": "demam"},
    {"label": "SYMPTOM", "pattern": "demam tinggi"},
    {"label": "SYMPTOM", "pattern": "kejang"},
    {"label": "SYMPTOM", "pattern": "kejang demam"},
    {"label": "SYMPTOM", "pattern": "batuk"},
    {"label": "SYMPTOM", "pattern": "sesak napas"},
    {"label": "SYMPTOM", "pattern": "sesak"},
    {"label": "SYMPTOM", "pattern": "mual"},
    {"label": "SYMPTOM", "pattern": "muntah"},
    {"label": "SYMPTOM", "pattern": "muntah darah"},
    {"label": "SYMPTOM", "pattern": "sakit kepala"},
    {"label": "SYMPTOM", "pattern": "sakit tenggorokan"},
    {"label": "SYMPTOM", "pattern": "sakit perut"},
    {"label": "SYMPTOM", "pattern": "pusing"},
    {"label": "SYMPTOM", "pattern": "lemas"},
    {"label": "SYMPTOM", "pattern": "lemah"},
    {"label": "SYMPTOM", "pattern": "nyeri"},
    {"label": "SYMPTOM", "pattern": "nyeri otot"},
    {"label": "SYMPTOM", "pattern": "nyeri kepala"},
    {"label": "SYMPTOM", "pattern": "koma"},
    {"label": "SYMPTOM", "pattern": "tidak sadar"},
    {"label": "SYMPTOM", "pattern": "gangguan kesadaran"},
    {"label": "SYMPTOM", "pattern": "gangguan pernapasan"},
    {"label": "SYMPTOM", "pattern": "radang otak"},
    {"label": "SYMPTOM", "pattern": "sulit bernapas"},
    {"label": "SYMPTOM", "pattern": "diare"},
    {"label": "SYMPTOM", "pattern": "disorientasi"},
    {"label": "SYMPTOM", "pattern": "kelelahan"},
    {"label": "SYMPTOM", "pattern": "mengantuk"},
    {"label": "SYMPTOM", "pattern": "tidak responsif"},
    {"label": "SYMPTOM", "pattern": "rasa sakit"},


    # ── SYMPTOM — Bahasa Inggris ──────────────────────────
    {"label": "SYMPTOM", "pattern": [{"LOWER": "fever"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "headache"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "cough"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "nausea"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "vomiting"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "fatigue"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "seizure"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "seizures"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "confusion"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "drowsiness"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "dizziness"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "disorientation"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "myalgia"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "difficulty"}, {"LOWER": "breathing"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "shortness"}, {"LOWER": "of"}, {"LOWER": "breath"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "chest"}, {"LOWER": "pain"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "muscle"}, {"LOWER": "pain"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "muscle"}, {"LOWER": "weakness"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "sore"}, {"LOWER": "throat"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "abdominal"}, {"LOWER": "pain"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "brain"}, {"LOWER": "swelling"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "loss"}, {"LOWER": "of"}, {"LOWER": "consciousness"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "altered"}, {"LOWER": "consciousness"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "respiratory"}, {"LOWER": "failure"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "respiratory"}, {"LOWER": "distress"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "respiratory"}, {"LOWER": "illness"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "respiratory"}, {"LOWER": "infection"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "respiratory"}, {"LOWER": "symptoms"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "acute"}, {"LOWER": "respiratory"}]},
    {"label": "SYMPTOM", "pattern": [{"LOWER": "inflammation"}, {"LOWER": "of"}, {"LOWER": "the"}, {"LOWER": "brain"}]},

    # ── LOCATION ──────────────────────────────────────────
    {"label": "LOCATION", "pattern": "Malaysia"},
    {"label": "LOCATION", "pattern": "India"},
    {"label": "LOCATION", "pattern": "Bangladesh"},
    {"label": "LOCATION", "pattern": "Singapore"},
    {"label": "LOCATION", "pattern": "Kerala"},
    {"label": "LOCATION", "pattern": "Indonesia"},
    {"label": "LOCATION", "pattern": "Philippines"},
    {"label": "LOCATION", "pattern": "Filipina"},
]

ruler.add_patterns(patterns)
print(f'Model NER berhasil dibuat!')
print(f'   Total patterns : {len(patterns)}')
print(f'   Pipeline aktif : {nlp.pipe_names}')

Model NER berhasil dibuat!
   Total patterns : 82
   Pipeline aktif : ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'entity_ruler']


In [30]:
# Uji coba model pada kalimat contoh
test_sentence = "Pasien di Kerala mengalami demam tinggi dan kejang akibat virus Nipah."
doc = nlp(test_sentence)

print(f'Kalimat uji: "{test_sentence}"')
print('\nEntitas terdeteksi:')
for ent in doc.ents:
    print(f'  [{ent.label_}] {ent.text}')

Kalimat uji: "Pasien di Kerala mengalami demam tinggi dan kejang akibat virus Nipah."

Entitas terdeteksi:
  [LOCATION] Kerala
  [SYMPTOM] demam tinggi
  [SYMPTOM] kejang
  [DISEASE] virus Nipah


In [31]:
def run_ner(text):
    """
    Menjalankan NER dan mengelompokkan entitas per label.
    Entitas yang sama hanya muncul sekali (menggunakan set).
    Contoh output: {'DISEASE': ['virus Nipah'], 'SYMPTOM': ['demam', 'kejang']}
    """
    doc = nlp(text)
    result = defaultdict(list)
    for ent in doc.ents:
        result[ent.label_].append(ent.text)
    return dict(result)

print('Menjalankan NER ke seluruh dataset...')
df_gabung['entities'] = df_gabung['text_clean'].apply(run_ner)

# Pisah ke kolom per label, entitas dipisah koma
df_gabung['DISEASE']  = df_gabung['entities'].apply(lambda x: ', '.join(set(x.get('DISEASE',  []))))
df_gabung['SYMPTOM']  = df_gabung['entities'].apply(lambda x: ', '.join(set(x.get('SYMPTOM',  []))))
df_gabung['LOCATION'] = df_gabung['entities'].apply(lambda x: ', '.join(set(x.get('LOCATION', []))))

print('NER selesai dijalankan!')
df_gabung[['text_clean', 'DISEASE', 'SYMPTOM', 'LOCATION']].head(10)

Menjalankan NER ke seluruh dataset...
NER selesai dijalankan!


,text_clean,DISEASE,SYMPTOM,LOCATION
0,Virus nipah (NiV) adalah virus zoonosis yang b...,"Virus nipah, NiV",,
1,Pertama kali diidentifikasi pada tahun 1998 di...,,,Malaysia
2,Menurut data dari World Health Organization (W...,"ensefalitis, virus nipah",radang otak,
3,"People with infection can develop a fever, and...",,"fever, confusion, cough, difficulty breathing,...",
4,Fruit bats of the Pteropodidae family are the ...,Nipah virus,,
5,Nipah virus was first identified in 1998 durin...,Nipah virus,,Malaysia
6,"In 1999, an outbreak was reported in Singapore...",,,"Malaysia, Singapore"
7,Fruit bats from the Pteropodidae family are co...,Nipah virus,,
8,Infection with Nipah virus does not appear to ...,Nipah virus,,
9,The incubation period a that is the time from ...,,,


In [32]:
# Visualisasi dengan warna custom per label
colors = {
    "DISEASE":  "#ff6b6b",
    "SYMPTOM":  "#feca57",
    "LOCATION": "#48dbfb"
}
options = {"ents": ["DISEASE", "SYMPTOM", "LOCATION"], "colors": colors}

# Uji beberapa kalimat sekaligus
test_sentences = [
    "Pasien di Kerala mengalami demam tinggi dan kejang akibat virus Nipah.",
    "Infeksi Nipah menyebabkan ensefalitis di Malaysia dan Bangladesh.",
    "Gejala awal berupa demam, mual, dan sakit kepala.",
]

for sent in test_sentences:
    doc = nlp(sent)
    print("Kalimat:", sent)
    print("Entitas:", [(ent.text, ent.label_) for ent in doc.ents])
    displacy.render(doc, style="ent", options=options, jupyter=True)
    print()

Kalimat: Pasien di Kerala mengalami demam tinggi dan kejang akibat virus Nipah.
Entitas: [('Kerala', 'LOCATION'), ('demam tinggi', 'SYMPTOM'), ('kejang', 'SYMPTOM'), ('virus Nipah', 'DISEASE')]



Kalimat: Infeksi Nipah menyebabkan ensefalitis di Malaysia dan Bangladesh.
Entitas: [('Nipah', 'DISEASE'), ('ensefalitis', 'DISEASE'), ('Malaysia', 'LOCATION'), ('Bangladesh', 'LOCATION')]



Kalimat: Gejala awal berupa demam, mual, dan sakit kepala.
Entitas: [('demam', 'SYMPTOM'), ('mual', 'SYMPTOM'), ('sakit kepala', 'SYMPTOM')]


In [33]:
# Statistik hasil NER
has_disease  = df_gabung['DISEASE'].str.len()  > 0
has_symptom  = df_gabung['SYMPTOM'].str.len()  > 0
has_location = df_gabung['LOCATION'].str.len() > 0
has_any      = has_disease | has_symptom | has_location

print('='*47)
print('        STATISTIK HASIL NER')
print('='*47)
print(f'  Total kalimat dataset   : {len(df_gabung)}')
print(f'  Mengandung DISEASE      : {has_disease.sum()} ({has_disease.mean()*100:.1f}%)')
print(f'  Mengandung SYMPTOM      : {has_symptom.sum()} ({has_symptom.mean()*100:.1f}%)')
print(f'  Mengandung LOCATION     : {has_location.sum()} ({has_location.mean()*100:.1f}%)')
print(f'  Memiliki >= 1 entitas   : {has_any.sum()} ({has_any.mean()*100:.1f}%)')
print(f'  Tidak ada entitas       : {(~has_any).sum()} ({(~has_any).mean()*100:.1f}%)')
print('='*47)

        STATISTIK HASIL NER
  Total kalimat dataset   : 871
  Mengandung DISEASE      : 374 (42.9%)
  Mengandung SYMPTOM      : 50 (5.7%)
  Mengandung LOCATION     : 156 (17.9%)
  Memiliki >= 1 entitas   : 455 (52.2%)
  Tidak ada entitas       : 416 (47.8%)


---
## 7. Evaluasi — Precision / Recall / F1

### Metodologi Evaluasi
Evaluasi dilakukan dengan membandingkan prediksi model terhadap **anotasi manual (ground truth)**. Ini adalah cara yang valid karena menguji model pada data yang belum pernah dilihat sebelumnya.

### Metrik yang Digunakan

| Metrik | Rumus | Interpretasi |
|---|---|---|
| **Precision** | TP / (TP + FP) | Dari semua yang diprediksi, berapa yang benar? |
| **Recall** | TP / (TP + FN) | Dari semua entitas benar, berapa yang berhasil ditemukan? |
| **F1 Score** | 2 × P × R / (P + R) | Keseimbangan antara precision dan recall |

Keterangan:
- **TP** (True Positive) = entitas diprediksi benar dan labelnya benar
- **FP** (False Positive) = diprediksi entitas, tapi salah
- **FN** (False Negative) = entitas nyata yang tidak terdeteksi

> ✏️ **Instruksi:** Lengkapi `ANNOTATED_DATA` di bawah dengan minimal **20–30 kalimat** beserta label manualnya dari dataset kamu. Semakin banyak, semakin valid hasilnya.

In [34]:
samples = df_gabung.sample(100, random_state=42)
for i, row in samples.iterrows():
    print(f"Index {i}: {row['text_clean']}")
    print()

Index 394: Anticipating the Nipah Virus, Central Java Tightens International Travel Surveillance - Kompas.id.

Index 66: Penularan virus Nipah dari manusia ke manusia telah dilaporkan dalam beberapa wabah.

Index 495: The sores burst and develop honey-colored crusts.

Index 67: Kontak langsung dengan cairan tubuh pasien meningkatkan risiko penularan.

Index 856: Regular hand washing should be carried out after caring for or visiting sick people along other preventive measures.

Index 766: In agricultural settings, improving farm biosecurity measures can reduce contact between bats and livestock, particularly pigs, which have historically served as an intermediate host during outbreaks.

Index 86: Penelitian vaksin virus Nipah masih dalam tahap pengembangan.

Index 804: Public health campaigns focused on food safety and avoiding bat habitats are essential for lowering these risks.

Index 606: Most experts do not classify the Nipah virus as airborne, though there is consensus that transm

In [50]:
ANNOTATED_DATA = [
    (
        "Nipah Virus Outbreak, Menginfeksi 5 Orang di India Termasuk Perawat - Liputan6.com.",
        {"DISEASE": ["Nipah Virus"], "SYMPTOM": [], "LOCATION": ["India"]}
    ),
    (
        "Two cases involved a single short exposure to an ill patient, including a rickshaw driver who transported a patient to the hospital.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "It has been reported in health-care settings and among family and caregivers of sick people through close contact.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Medication Ribavirin has been studied in a small number of people.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "The World Health Organization classifies Nipah virus as a priority pathogen.",
        {"DISEASE": ["Nipah virus"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Numerous disease outbreaks caused by the Nipah virus have occurred in India, Malaysia, and Singapore.",
        {"DISEASE": ["Nipah virus"], "SYMPTOM": [], "LOCATION": ["India", "Malaysia", "Singapore"]}
    ),
    (
        "Virus Nipah, Kenali Gejalanya, Cegah Penularannya - Indonesia Baik.",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Setelah berikatan dengan reseptor, pori fusi terbentuk pada selubung sel epitel, memungkinkan NiV berinvasi ke dalam sel.",
        {"DISEASE": ["NiV"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "BSL 2 fasilitas memadai jika virus dapat diinaktivasi terlebih dahulu pada saat pengambilan spesimen.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Penanganan pasien virus Nipah berfokus pada perawatan suportif.",
        {"DISEASE": ["virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Berikut Catatan Wabah Sejak 1998 Liputan6.com",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Not everyone with a UTI has symptoms, but common symptoms include a frequent urge to urinate and pain or burning when urinating.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "A lock ( ) or https:// means you've safely connected to the .gov website.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Waspada Penyebaran Virus Nipah, Barantin Perketat Lalu Lintas Komoditas Hewan dan Tumbuhan - Kementerian Komunikasi dan Digital.",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "75 of patients were either hospital staff or had visited one of the other patients in hospital, indicating person-to-person transmission.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Isolation of patients helps prevent further transmission.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Waspada Virus Nipah 2026: Gejala, Cara Pencegahan, dan Situasi Terkini di Asia Universitas Negeri Surabaya",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Diarrhea is uncommon, and vomiting is not usually severe.",
        {"DISEASE": [], "SYMPTOM": ["vomiting"], "LOCATION": []}
    ),
    (
        "Healthcare workers must use appropriate protective equipment.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "In Bangladesh there were also outbreaks in subsequent years.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": ["Bangladesh"]}
    ),
    (
        "Soekarno-Hatta Airport Intensifies Screening to Block Nipah Virus Entry - Polri.",
        {"DISEASE": ["Nipah Virus"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Measures such as covering animal feed, preventing bat access to livestock enclosures, and maintaining sanitary farm environments help reduce the risk of viral transmission.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "In vitro studies and animal studies have shown conflicting results in the efficacy of ribavirin against NiV and Hendra, with some studies showing effective inhibition of viral replication in cell lines, whereas some studies in animal models showed that ribavirin treatment only delayed but did not prevent death after NiV or Hendra virus infection.",
        {"DISEASE": ["NiV"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Penularan virus Nipah dari manusia ke manusia telah dilaporkan dalam beberapa wabah.",
        {"DISEASE": ["virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Penggunaan alat pelindung diri sangat penting dalam mencegah penularan.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "This region covers just 5 of the Earth's total land area, yet it is home to 26 of the global population.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Cegah Penularan Virus Nipah, Kakes Koopsau Tekankan Pola Hidup Bersih dan Sehat - TNI AU.",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Virus Nipah dapat berpindah dari kelelawar ke manusia secara tidak langsung.",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Bantuan pernapasan, seperti oksigen atau ventilator, jika terjadi masalah pernapasan.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Kontak langsung dengan cairan tubuh pasien meningkatkan risiko penularan.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Kasus virus Nipah juga pernah dilaporkan di Singapura.",
        {"DISEASE": ["virus Nipah"], "SYMPTOM": [], "LOCATION": ["Singapore"]}
    ),
    (
        "In Bangladesh and India, the disease has been linked to consumption of raw date palm sap (toddy), eating of fruits partially consumed by bats, and using water from wells inhabited by bats.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": ["Bangladesh", "India"]}
    ),
    (
        "Penelitian vaksin virus Nipah masih dalam tahap pengembangan.",
        {"DISEASE": ["virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Ancaman Virus Nipah India Menguat, Indonesia Waspada - Beranda Post.",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": ["India", "Indonesia"]}
    ),
    (
        "Virus Nipah merupakan tipe virus RNA yang berasal dari genus Henipavirus.",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Nipah Virus Outbreak dan Risiko Penularan Antar Manusia, Apakah Ada Vaksin atau Obat Virus Nipah?",
        {"DISEASE": ["Nipah Virus", "Virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Pemerintah memiliki peran penting dalam respons kesehatan masyarakat.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Karantina pasien diperlukan untuk mencegah penyebaran virus Nipah.",
        {"DISEASE": ["virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "BRIN Temukan RNA Virus Nipah pada Kelelawar di Indonesia infonasional.com",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": ["Indonesia"]}
    ),
    (
        "Acquired immunodeficiency syndrome (AIDS) is a chronic, potentially life-threatening condition caused by the human immunodeficiency virus (HIV).",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Stricter implementation of preventive measures to reduce COVIDa19 caseload will also reduce the strain on the healthcare industry.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Rare brain-damaging Nipah virus kills 10 in India, prompts rush to hospitals The Straits Times",
        {"DISEASE": ["Nipah virus"], "SYMPTOM": [], "LOCATION": ["India"]}
    ),
    (
        "In Singapore, 11 cases, including one death, occurred in slaughterhouse workers exposed to pigs imported from the affected Malaysian farms.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": ["Singapore"]}
    ),
    (
        "Infeksi Virus Nipah: Asal Negara, Gejala, Adakah Kasusnya di Indonesia?",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": ["Indonesia"]}
    ),
    (
        "Health systems must remain vigilant against emerging infections.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "No further clinical studies with ribavirin have been conducted, and research in animal models has not demonstrated its effectiveness against Nipah or Hendra virus infections.",
        {"DISEASE": ["Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Bangladesh merupakan salah satu negara dengan kasus virus Nipah yang berulang.",
        {"DISEASE": ["virus Nipah"], "SYMPTOM": [], "LOCATION": ["Bangladesh"]}
    ),
    (
        "Gastroenteritis is an inflammation of the digestive tract, particularly the stomach, and large and small intestines.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Nipah virus is one of several viruses identified by the World Health Organization (WHO) as a potential cause of future epidemics in a new plan developed after the Ebola epidemic for urgent research and development toward new diagnostic tests, vaccines and medicines.",
        {"DISEASE": ["Nipah virus"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Berikut Catatan Wabah Sejak 1998 - Liputan6.com.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Typhoid fever has an insidious onset, with fever, headache, constipation, malaise, chills, and muscle pain.",
        {"DISEASE": [], "SYMPTOM": ["fever", "headache", "muscle pain"], "LOCATION": []}
    ),
    (
        "A rare form of liver inflammation caused by infection with the hepatitis E virus (HEV).",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "In India, outbreaks are periodically reported in several parts of the country, including the latest one in 2026.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": ["India"]}
    ),
    (
        "Although this trial was small (30 participants) and unable to evaluate protective efficacy, the safety and tolerability demonstrated in this study make m102.4 one of the most promising therapeutic options for the treatment of patients with HNV exposure.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Waspadai Virus Nipah, Kemenkes Awasi Ketat Semua Pelaku Perjalanan Internasional - Kompas.id.",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "doi:10.1186/s40824-016-0087-x.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "WHO Ungkap Gejala, Pola Penularan, dan Risiko Penyakit Virus Nipah - InfoPublik.",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "An adverse drug reaction (ADR) is an injury caused by taking medication.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Waspada Penyebaran Virus Nipah, Barantin Perketat Lalu Lintas Komoditas Hewan dan Tumbuhan Kementerian Komunikasi dan Digital",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "The sores burst and develop honey-colored crusts.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "In January 2026, health authorities in West Bengal, India, moved to contain a confirmed Nipah virus outbreak after reporting five confirmed cases, including healthcare workers.",
        {"DISEASE": ["Nipah virus"], "SYMPTOM": [], "LOCATION": ["India"]}
    ),
    (
        "Nipah virus (Henipavirus nipahense) is a bat-borne, zoonotic virus that causes Nipah virus infection in humans and other animals, a disease with a very high case fatality rate (4075 ).",
        {"DISEASE": ["Nipah virus"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "It occurs when the protective cartilage that cushions the ends of your bones wears down over time.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "m102.4 has been used in people on a compassionate use basis in Australia, and was in pre-clinical development in 2013.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Ini Gejala dan Bahayanya - Mitra Keluarga.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Transparansi data mendukung upaya pengendalian wabah.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Geographic distribution Nipah virus has been isolated from Lyle's flying fox (Pteropus lylei) in Cambodia and viral RNA found in urine and saliva from P.",
        {"DISEASE": ["Nipah virus"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Wabah Virus Nipah yang Berulang - Kompas.id.",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Intracerebral hemorrhage (ICH) is when blood suddenly bursts into brain tissue, causing damage to your brain.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Indonesia memiliki tingkat interaksi manusia dan satwa liar yang tinggi, termasuk keberadaan kelelawar sebagai reservoir alami virus",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": ["Indonesia"]}
    ),
    (
        "Tuberculosis (TB) is an infectious disease usually caused by Mycobacterium tuberculosis (MTB) bacteria.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "The yellowing extends to other tissues and body fluids.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "The common cold is a viral infection of your nose and throat (upper respiratory tract).",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Dalam proses ini, NiV memanfaatkan protein F dan G untuk berinteraksi dengan reseptor permukaan sel epitel Ephrin B2/B3 di saluran pernapasan bagian bawah",
        {"DISEASE": ["NiV"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Hypoglycemia is often related to diabetes treatment.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Virus Nipah Mengintai, Peneliti BRIN Jelaskan Risiko dan Upaya Kesiapsiagaan - BRIN - Badan Riset dan Inovasi Nasional.",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "India Konfirmasi Kasus Baru Virus Nipah, Negara-Negara Tetangga Tingkatkan Kewaspadaan Kompas.tv",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": ["India"]}
    ),
    (
        "In some rare cases incubation of up to 45 days has been reported.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "In severe cases patients may develop encephalitis, which can lead to dizziness, drowsiness, altered consciousness and seizures.",
        {"DISEASE": ["encephalitis"], "SYMPTOM": ["dizziness", "drowsiness", "altered consciousness", "seizures"], "LOCATION": []}
    ),
    (
        "Health Ministry Urges Vigilance Against Nipah Virus Following Outbreak in India - Polri.",
        {"DISEASE": ["Nipah Virus"], "SYMPTOM": [], "LOCATION": ["India"]}
    ),
    (
        "There are three unique folding patterns of the heads, resulting in a 2-up/2-down configuration where two heads are positioned distal to the virus, and two heads are proximal.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Virus Nipah diklasifikasikan oleh CDC sebagai agen Kategori C.",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Prevention Is the Only Effective Way to Overcome the Nipah Virus Kompas.id",
        {"DISEASE": ["Nipah Virus"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Fungi can live in the air, soil, water, and plants.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "The virus was probably contracted from drinking date palm juice contaminated by fruit bat droppings or saliva.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Tightening Entry Points to Anticipate Nipah Virus Kompas.id",
        {"DISEASE": ["Nipah Virus"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Dokter RSA UGM Sampaikan Tanda Gejala Awal Tertular Virus Nipah - Universitas Gadjah Mada.",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "After undergoing treatment for 54 days at a private hospital, the 23-year-old student was discharged.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Regular hand washing should be carried out after caring for or visiting sick people along other preventive measures.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Deaths from reactivation of latent Nipah virus have been reported.",
        {"DISEASE": ["Nipah virus"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Gloves and other protective clothing should be worn while handling sick animals such as pigs or horses, and during slaughtering and culling procedures.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "BRIN Temukan RNA Virus Nipah pada Kelelawar di Indonesia - infonasional.com.",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": ["Indonesia"]}
    ),
    (
        "Preventive measures include avoiding exposure to bats and infected animals such as pigs, and not drinking raw date palm sap.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Risk factors The risk of exposure is high for hospital workers and caretakers of those infected with the virus.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "In the 199899 Nipah virus outbreak in Malaysia, 140 patients received ribavirin, with their outcomes assessed against 54 historical controls who either lacked access to the drug or declined treatment.",
        {"DISEASE": ["Nipah virus"], "SYMPTOM": [], "LOCATION": ["Malaysia"]}
    ),
    (
        "Hindari konsumsi daging kelelawar atau daging hewan ternak yang dimasak kurang matang.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Laporan setempat menyebutkan bahwa hampir 100 orang, yang semuanya telah melakukan kontak dengan pasien telah dikarantina dan berada di bawah pengawasan.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Rapid reporting of suspected cases improves outbreak response.",
        {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "Virus Nipah, Ancaman Zoonosis yang Perlu Diwaspadai - Kompaspedia.",
        {"DISEASE": ["Virus Nipah"], "SYMPTOM": [], "LOCATION": []}
    ),
    (
        "India Klaim Berhasil Mengendalikan Wabah Nipah di Wilayahnya - MetroTVNews.com.",
        {"DISEASE": ["Nipah"], "SYMPTOM": [], "LOCATION": ["India"]}
    ),
]

print(f'Jumlah data anotasi ground truth: {len(ANNOTATED_DATA)} kalimat')

Jumlah data anotasi ground truth: 100 kalimat


In [51]:
def evaluate_ner(annotated_data, model_func):
    """
    Menghitung Precision, Recall, dan F1 per label.
    Perbandingan dilakukan case-insensitive (lowercase).
    """
    labels = ['DISEASE', 'SYMPTOM', 'LOCATION']
    stats  = {l: {'TP': 0, 'FP': 0, 'FN': 0} for l in labels}

    for text, gold in annotated_data:
        pred = model_func(text)
        for label in labels:
            gold_set = set(e.lower() for e in gold.get(label, []))
            pred_set = set(e.lower() for e in pred.get(label, []))
            stats[label]['TP'] += len(gold_set & pred_set)
            stats[label]['FP'] += len(pred_set - gold_set)
            stats[label]['FN'] += len(gold_set - pred_set)

    results = {}
    for label in labels:
        tp = stats[label]['TP']
        fp = stats[label]['FP']
        fn = stats[label]['FN']
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        results[label] = {
            'Precision': round(precision, 3),
            'Recall':    round(recall,    3),
            'F1 Score':  round(f1,        3),
            'TP': tp, 'FP': fp, 'FN': fn
        }
    return results


def spacy_predict(text):
    doc = nlp(text)
    result = defaultdict(list)
    for ent in doc.ents:
        result[ent.label_].append(ent.text)
    return dict(result)


# Jalankan evaluasi
eval_results = evaluate_ner(ANNOTATED_DATA, spacy_predict)
df_eval = pd.DataFrame(eval_results).T

print('=' * 57)
print('      HASIL EVALUASI — spaCy Rule-Based NER')
print('=' * 57)
print(df_eval[['Precision', 'Recall', 'F1 Score', 'TP', 'FP', 'FN']].to_string())
print('=' * 57)

      HASIL EVALUASI — spaCy Rule-Based NER
          Precision  Recall  F1 Score    TP   FP   FN
DISEASE       1.000   1.000     1.000  47.0  0.0  0.0
SYMPTOM       1.000   1.000     1.000   8.0  0.0  0.0
LOCATION      0.957   0.957     0.957  22.0  1.0  1.0


In [52]:
for text, gold in ANNOTATED_DATA:
    pred = spacy_predict(text)
    gold_set = set(e.lower() for e in gold.get('SYMPTOM', []))
    pred_set = set(e.lower() for e in pred.get('SYMPTOM', []))
    fn = gold_set - pred_set
    if fn:
        print(f'Kalimat : {text[:80]}')
        print(f'FN SYMPTOM : {fn}')
        print()

In [53]:
# Macro Average
avg_p  = df_eval['Precision'].mean()
avg_r  = df_eval['Recall'].mean()
avg_f1 = df_eval['F1 Score'].mean()

print('\n📊 Macro Average (rata-rata semua label):')
print(f'   Precision : {avg_p:.3f}')
print(f'   Recall    : {avg_r:.3f}')
print(f'   F1 Score  : {avg_f1:.3f}')

print('\n💡 Interpretasi per label:')
for label, row in df_eval.iterrows():
    f1 = row['F1 Score']
    if f1 >= 0.8:
        status = '🟢 Baik'
    elif f1 >= 0.6:
        status = '🟡 Cukup'
    else:
        status = '🔴 Perlu ditingkatkan'
    print(f'   {label:10s} F1={f1:.3f}  {status}')


📊 Macro Average (rata-rata semua label):
   Precision : 0.986
   Recall    : 0.986
   F1 Score  : 0.986

💡 Interpretasi per label:
   DISEASE    F1=1.000  🟢 Baik
   SYMPTOM    F1=1.000  🟢 Baik
   LOCATION   F1=0.957  🟢 Baik


In [54]:
for text, gold in ANNOTATED_DATA:
    pred = spacy_predict(text)
    gold_set = set(e.lower() for e in gold.get('SYMPTOM', []))
    pred_set = set(e.lower() for e in pred.get('SYMPTOM', []))
    fp = pred_set - gold_set
    if fp:
        print(f'Kalimat : {text[:80]}')
        print(f'FP : {fp}')
        print()

In [55]:
for text, gold in ANNOTATED_DATA:
    pred = spacy_predict(text)
    gold_set = set(e.lower() for e in gold.get('DISEASE', []))
    pred_set = set(e.lower() for e in pred.get('DISEASE', []))
    fp = pred_set - gold_set
    fn = gold_set - pred_set
    if fp or fn:
        print(f'Kalimat : {text[:80]}')
        print(f'FP : {fp}')
        print(f'FN : {fn}')
        print()

Evaluasi dilakukan pada 100 kalimat yang diambil secara acak dari dataset menggunakan random sampling (random_state=42). Model berhasil mencapai Macro F1 sebesar 0.986 dengan DISEASE F1=1.000, SYMPTOM F1=1.000, dan LOCATION F1=0.957. Label LOCATION memiliki F1 terendah karena model rule-based tidak dapat membedakan nama entitas yang muncul sebagai nama media atau organisasi (contoh: 'CNBC Indonesia', 'Media Indonesia') dengan lokasi geografis sesungguhnya — ini merupakan keterbatasan inherent dari pendekatan rule-based yang tidak mempertimbangkan konteks kalimat secara menyeluruh. Sebagai arah pengembangan ke depan, penggunaan model berbasis machine learning seperti BERT dapat meningkatkan performa terutama untuk label LOCATION, karena model tersebut mampu memahami konteks kalimat secara menyeluruh sehingga dapat membedakan nama entitas yang ambigu.

---
## 8. Simpan Model & Dataset Final

Setelah seluruh proses selesai, model dan dataset disimpan untuk:
- Digunakan kembali di aplikasi Streamlit
- Dokumentasi dan submission tugas
- Reprodusibilitas eksperimen di kemudian hari

In [61]:
# Simpan model spaCy ke disk
nlp.to_disk('nipah_ner_model')
print('Model tersimpan di folder: nipah_ner_model/')

Model tersimpan di folder: nipah_ner_model/


In [60]:
# Sesuaikan path penyimpanan dengan Google Drive kamu
SAVE_PATH = '/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/Dataset_Nipah_Final.csv'

df_gabung[['id', 'text_clean', 'DISEASE', 'SYMPTOM', 'LOCATION']].to_csv(SAVE_PATH, index=False)

print(f'✅ Dataset final tersimpan!')
print(f'   Path   : {SAVE_PATH}')
print(f'   Baris  : {len(df_gabung)}')
print(f'   Kolom  : id, text_clean, DISEASE, SYMPTOM, LOCATION')

✅ Dataset final tersimpan!
   Path   : /content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/Dataset_Nipah_Final.csv
   Baris  : 871
   Kolom  : id, text_clean, DISEASE, SYMPTOM, LOCATION


In [63]:
import shutil
shutil.make_archive('nipah_ner_model', 'zip', 'nipah_ner_model')
print('✅ Model di-zip: nipah_ner_model.zip — siap didownload!')
shutil.make_archive(
    '/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/nipah_ner_model',
    'zip',
    'nipah_ner_model'
)
print('✅ Model.zip berhasil disimpan ke Drive!')

✅ Model di-zip: nipah_ner_model.zip — siap didownload!
✅ Model.zip berhasil disimpan ke Drive!
